# R0 修复接线验证（只验 2 分子，约 10 分钟，T4）。按顺序跑，每格末尾有 EXPECT 行，对不上就停、贴回来。
目标：验证 `np.str_` 落盘、`tmp.npz` 后缀、`run_index` 按 exp 合并三处修复。不跑全量 250：全程带 `--mol-limit 2`，**不要去掉（去掉会跑全量 250）**。不读不评 heldout。只做接线验证，不做任何 MRR 结论。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# EXPECT: Mounted at /content/drive


In [ ]:
!tar xzf /content/drive/MyDrive/r0-fixcheck.tar.gz -C /content && (test -d /content/r0-fixcheck || mv /content/artifacts/r0-fixcheck /content/r0-fixcheck) && ls /content/r0-fixcheck
# EXPECT: 看到 inputs.json install.sh run_g1.py g1_worker.py weights code README.md


In [ ]:
!cd /content/r0-fixcheck && bash install.sh
# EXPECT: 末尾 install done. + cuda True + Tesla T4


In [ ]:
!cd /content/r0-fixcheck && python3 run_g1.py --exps g1e --dry-run
# EXPECT: 两格 ok-dry-run，无 Traceback。只验接线，不跑模型。（默认 --mol-limit 2）


In [ ]:
!cd /content/r0-fixcheck && rm -f scores/g1e-2* scores/run_index.json && python3 run_g1.py --exps g1e-2 --mol-limit 2 --rerun
# EXPECT: 末尾 cell g1e-2 status=ok（不是ok-dry-run）。不要去掉 --mol-limit 2（去掉会跑全量250）。约5-8分钟。


In [ ]:
!cd /content/r0-fixcheck && rm -f scores/g1e-4* && python3 run_g1.py --exps g1e-4 --mol-limit 2 --rerun
# EXPECT: 末尾 cell g1e-4 status=ok（复用缓存，增量小）。注意本格不删 run_index.json（为验证按exp合并）。不要去掉 --mol-limit 2。


In [ ]:
!cd /content/r0-fixcheck && ls cache/iceberg/*.npz
# EXPECT: 两格各至少1个 .npz（以分子mid命名）。没有就停。
!cd /content/r0-fixcheck && for f in cache/iceberg/*.npz; do python3 -c "import numpy as np,json,sys; p=sys.argv[1]; z=np.load(p,allow_pickle=False); keys=json.loads(str(z['_keys'].tolist())); missing=[k for k in keys if 'p_'+k+'_mz' not in z.files or 'p_'+k+'_it' not in z.files]; print(p,'keys=',len(keys),'missing=',len(missing)); assert keys and not missing" "$f"; done
# EXPECT: 每个npz keys>0且missing=0（disk_load逐键读回成功即停）。有missing就停。
!cd /content/r0-fixcheck && python3 -c "import json,glob; ms=sorted(glob.glob('scores/*.meta.json')); print(ms); [print(m, json.load(open(m)).get('status'), 'n_scored=', json.load(open(m)).get('n_scored'), 'errs=', len(json.load(open(m)).get('errors',[]))) for m in ms]"
# EXPECT: 两格status=ok（非dry-run），n_scored>0。对不上就停。
!cd /content/r0-fixcheck && (grep -h 'unicode_\|replace' scores/*.meta.json || echo no-unicode/replace-in-meta-OK)
# EXPECT: no-unicode/replace-in-meta-OK（印出匹配行就停）。
!cd /content/r0-fixcheck && cat scores/run_index.json && python3 -c "import json; d=json.load(open('scores/run_index.json')); exps=[c['exp'] for c in d['cells']]; print(exps,'seconds=',d['seconds']); assert 'g1e-2' in exps and 'g1e-4' in exps,'index merge failed'"
# EXPECT: run_index.json同时含g1e-2与g1e-4两格（若只有一格，合并失败，停）。
!cd /content/r0-fixcheck && ls -R cache > cache_manifest.txt && tar czf r0-fixcheck-out.tar.gz scores/*.meta.json scores/run_index.json cache_manifest.txt && ls -lh r0-fixcheck-out.tar.gz
# EXPECT: r0-fixcheck-out.tar.gz若干KB（只含meta+index+清单，无大峰文件）。去Files面板下载发回。
